# Agent 1 — A Model in a Loop

A chatbot answers from memory; an agent goes and looks. This notebook builds
the plan-act-observe loop with a **scripted planner**, so you can see every
turn with nothing hidden. Lesson 2 swaps in a real model.

Run every cell, top to bottom. Change things and rerun — the mini-web is
yours to break.

In [ ]:
# The mini-web: nine pages about the (fictional) Riverside Community Garden.
# Small enough to read whole, real enough to research. One page is wrong on purpose.
MINIWEB = {
 "riverside-garden.org/about": {"date": "2026-05-10", "title": "Our garden today",
  "text": "The Riverside Community Garden has 60 plots and 48 member families. "
          "We grow vegetables for members and donate surplus to the food pantry."},
 "riverside-garden.org/history": {"date": "2023-05-02", "title": "Our history",
  "text": "Founded in 2019 with a dozen beds. The sign by the gate lists 48 plots, "
          "painted when we finished the 2023 season."},
 "riverside-garden.org/join": {"date": "2026-06-01", "title": "Join us",
  "text": "Want a plot? The waitlist currently holds 22 families. Members pay a "
          "small annual fee and share watering duties."},
 "lakeview-news.com/garden-expands": {"date": "2026-04-20", "title": "Garden adds 12 plots",
  "text": "The Riverside Community Garden completed its expansion this spring, "
          "taking the garden from 48 plots to 60. Organizers credit a city grant."},
 "lakeview-news.com/roundup-2023": {"date": "2023-09-15", "title": "Community roundup",
  "text": "At the Riverside garden, 31 member families closed out the 2023 season "
          "with a harvest festival."},
 "cityparks.gov/report-2026": {"date": "2026-03-14", "title": "Community garden census",
  "text": "Riverside Community Garden: 60 plots, 48 member families, established "
          "2019. Census conducted March 2026."},
 "cityparks.gov/grants-2025": {"date": "2025-11-08", "title": "2025 grant awards",
  "text": "Riverside Community Garden: $15,000 for expansion. The site's land "
          "lease with the parks department runs through 2028."},
 "gardenblog.example.com/visit": {"date": "2026-02-02", "title": "A visit to Riverside",
  "text": "Lovely afternoon at Riverside! I heard they have 600 plots now, which "
          "explains the crowds. The tomatoes were spectacular."},
 "gardenblog.example.com/opinion": {"date": "2026-01-05", "title": "Why gardens matter",
  "text": "Community gardens are the beating heart of a neighborhood. Riverside "
          "is a treasure and everyone loves it."},
}

import re as _re, collections as _c
def _words(text):
    return set(w for w in _re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2)
_DF = _c.Counter()                       # in how many pages does each word appear?
for _p in MINIWEB.values():
    for _w in _words(_p["title"] + " " + _p["text"]):
        _DF[_w] += 1

def search(query):
    """Score pages by shared words, each weighted by rarity (1/pages-containing-it).
    'riverside' is on every page and says nothing; 'waitlist' is on one and says a lot."""
    qwords = _words(query)
    scored = []
    for url, page in MINIWEB.items():
        shared = qwords & _words(page["title"] + " " + page["text"])
        scored.append((sum(1.0 / _DF[w] for w in shared), url, page["title"]))
    scored.sort(reverse=True)
    return [(url, title) for score, url, title in scored[:3] if score > 0.3]

def fetch(url):
    """Return a page's text with its receipt (url and date) attached."""
    page = MINIWEB[url]
    return {"url": url, "date": page["date"], "text": page["text"]}

print(f"{len(MINIWEB)} pages online.")
print("search('riverside garden plots') ->")
for url, title in search("riverside garden plots"):
    print("  ", url, "-", title)

## The dispatch table — your code is the hands

The planner (scripted today, a model tomorrow) can only *request* actions.
This dictionary decides what can actually run. Nothing outside it exists,
no matter who asks.

In [ ]:
TOOLS = {"search": search, "fetch": lambda url: fetch(url)["text"] + "  (published " + fetch(url)["date"] + ")"}

def dispatch(action, arg):
    if action not in TOOLS:
        return f"ERROR: no tool named {action!r}. Available: {list(TOOLS)}"
    return TOOLS[action](arg)

print(dispatch("search", "riverside garden plots"))
print(dispatch("delete_files", "/"))   # the safety property, demonstrated

## A scripted planner

A real model decides its next action by reading the conversation. To see the
loop's shape first, we script those decisions: a list of (action, argument)
pairs ending in an answer. The LOOP is identical either way — only the
planner changes in lesson 2.

In [ ]:
QUESTION = "How many plots does the Riverside Community Garden have?"

SCRIPT = [
    ("search", "riverside community garden plots"),
    ("fetch",  "cityparks.gov/report-2026"),
    ("answer", "The Riverside Community Garden has 60 plots, per the city "
               "parks census of March 2026 [cityparks.gov/report-2026]."),
]

def run_scripted_agent(question, script):
    transcript = [f"QUESTION: {question}"]
    for action, arg in script:
        if action == "answer":
            transcript.append(f"MODEL:  ANSWER: {arg}")
            break
        transcript.append(f"MODEL:  ACTION: {action}({arg!r})")
        result = dispatch(action, arg)
        transcript.append(f"CODE:   {result}")
    return transcript

for line in run_scripted_agent(QUESTION, SCRIPT):
    print(line)
    print()

## What just happened

- **The model never touched the world.** Every MODEL line is a request in
  text; every CODE line is your dispatch running a real function.
- **The evidence arrived during the loop.** The answer cites a URL and a
  date because the fetch put them into the transcript — nothing was
  remembered from training.
- **The loop is tiny.** Plan, act, observe, repeat. Everything else in this
  course refines one of those four words.

## Try it

1. Change the script to fetch `lakeview-news.com/garden-expands` instead.
   Does the news page support the same answer?
2. Add a step that fetches the blog post (`gardenblog.example.com/visit`).
   Its number disagrees with the census — write the answer you'd give
   after seeing both, and keep that thought: it's lesson 6's whole job.
3. **Build turn-in:** annotate a full transcript with M (model) or C (code)
   per line, plus two sentences: what breaks without the dispatch table?